In [ ]:
# ============================================================================
# 02_collect_prices_and_factors.ipynb
# Download free monthly equity prices (yfinance) for the analysis universe and
# the Fama-French 3 factors plus the momentum factor (Kenneth French Data
# Library). These feed the CAPM / FF3 / Carhart portfolio regressions
# (paper Eq. 12-14).
#
# NOTE ON DATA QUALITY: free price feeds are NOT survivorship-bias-free and are
# not point-in-time correct. Delisted names may be missing, which can inflate
# long-short returns. This limitation is documented and discussed in the paper;
# the accounting-based regressions in notebook 03 are less exposed to it.
# ============================================================================

In [ ]:
# --- Imports and configuration -------------------------------------------
import urllib.request, io, zipfile, os, time
import pandas as pd
import numpy as np
import yfinance as yf

DATA_DIR = "../data"
START, END = "2013-01-01", "2021-12-31"

In [ ]:
# --- Download monthly adjusted-close prices (batched, incremental) --------
# We cap the universe for runtime feasibility; the cap is deterministic
# (alphabetical) so the sample is reproducible. Raise/remove the cap if you
# have time to download the full universe.
uni = pd.read_csv(f"{DATA_DIR}/universe_tickers.csv")["ticker"].tolist()
uni = [t for t in uni if isinstance(t, str) and t.isalpha() and len(t) <= 5]
uni = uni[:900]

closes = {}
BATCH = 40
for i in range(0, len(uni), BATCH):
    chunk = uni[i:i+BATCH]
    try:
        d = yf.download(chunk, start=START, end=END, interval="1mo",
                        progress=False, auto_adjust=True, threads=False)
        cl = d["Close"] if isinstance(d.columns, pd.MultiIndex) else d[["Close"]]
        for t in chunk:
            if t in cl.columns:
                s = cl[t].dropna()
                if len(s) > 24:          # need a reasonable history
                    closes[t] = s
    except Exception as e:
        print("batch fail", i, str(e)[:50])
    print(f"{i+len(chunk)}/{len(uni)} kept={len(closes)}")
    time.sleep(0.5)

px = pd.DataFrame(closes).sort_index()
px.to_csv(f"{DATA_DIR}/monthly_prices.csv")
print("Saved prices:", px.shape)

In [ ]:
# --- Download Fama-French 3 factors + momentum (Kenneth French) -----------
def french_text(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=30).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    return z.read(z.namelist()[0]).decode("latin-1")

ff  = french_text("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip")
mom = french_text("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip")

In [ ]:
# --- Parse the monthly factor blocks -------------------------------------
# Monthly rows are 6-digit YYYYMM keys; we keep those and drop the annual block.
def parse_ff3(txt):
    rows = []
    for line in txt.splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) >= 5 and p[0].isdigit() and len(p[0]) == 6:
            rows.append(p[:5])
    df = pd.DataFrame(rows, columns=["ym", "MktRF", "SMB", "HML", "RF"]).set_index("ym")
    return df.astype(float)

def parse_mom(txt):
    rows = []
    for line in txt.splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) >= 2 and p[0].isdigit() and len(p[0]) == 6:
            rows.append(p[:2])
    return pd.DataFrame(rows, columns=["ym", "WML"]).set_index("ym").astype(float)

fac = parse_ff3(ff).join(parse_mom(mom), how="inner")
fac.index = pd.to_datetime(fac.index, format="%Y%m").to_period("M").to_timestamp()
fac = fac / 100.0                 # percent -> decimal
fac = fac.loc["2013":"2021"]
fac.to_csv(f"{DATA_DIR}/ff_factors.csv")
print("Saved factors:", fac.shape)
fac.head(3)